# Test Hugging Face VideoMAE HD-EPIC sur vidéos culinaires

Objectif : charger le checkpoint Hugging Face `cstokl3/videomae-hdepic-verb-recognition`, faire l'inférence sur une vidéo découpée en segments, puis produire :

- une vidéo annotée avec verbe, objet, main et scores ;
- un fichier CSV avec les prédictions segment par segment.

À exécuter sur **Google Colab avec GPU** : `Exécution > Modifier le type d'exécution > GPU`.

## 1. Vérifier le GPU

In [2]:
!nvidia-smi

Wed Jul  8 13:32:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Installer les dépendances

In [3]:
!pip install -q transformers accelerate huggingface_hub opencv-python pillow pandas tqdm einops

## 3. Imports et configuration

In [4]:
import os
import glob
import json
import math
import cv2
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from base64 import b64encode
from IPython.display import HTML, display
from google.colab import files
from huggingface_hub import hf_hub_download, list_repo_files
from transformers import VideoMAEModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "cstokl3/videomae-hdepic-verb-recognition"
BASE_MODEL_ID = "MCG-NJU/videomae-base-finetuned-ssv2"

print("Device utilisé :", DEVICE)

Device utilisé : cuda


## 4. Télécharger le checkpoint Hugging Face

Le fichier principal est `best.pt`. On affiche aussi la liste des fichiers disponibles pour faciliter le debug.

In [5]:
print("Fichiers du repo Hugging Face :")
try:
    for f in list_repo_files(MODEL_ID):
        print("-", f)
except Exception as e:
    print("Impossible de lister les fichiers :", e)

ckpt_path = hf_hub_download(repo_id=MODEL_ID, filename="best.pt")
print("Checkpoint téléchargé :", ckpt_path)

Fichiers du repo Hugging Face :


- .gitattributes
- README.md
- best.pt
- yolo_epic_kitchens.pt


best.pt:   0%|          | 0.00/1.04G [00:00<?, ?B/s]

Checkpoint téléchargé : /root/.cache/huggingface/hub/models--cstokl3--videomae-hdepic-verb-recognition/snapshots/05b64c18afca6a79f8bf817f247006270d79a71a/best.pt


## 5. Charger les noms de classes depuis le checkpoint

In [6]:
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print("Clés du checkpoint :", ckpt.keys())

verb_names = ckpt.get("verb_names", None)
noun_names = ckpt.get("noun_names", None)
hand_names = ckpt.get("hand_names", None)

if verb_names is None:
    verb_names = ['take', 'put', 'wash', 'open', 'close', 'turn-on', 'cut', 'turn-off', 'pour', 'mix', 'move', 'remove', 'throw', 'dry', 'shake', 'scoop', 'adjust', 'squeeze', 'press', 'flip', 'turn', 'check', 'scrub', 'pull', 'pat', 'lift', 'hold', 'drop', 'transition', 'reach']

if hand_names is None:
    hand_names = ["left", "right", "both"]

n_verb_classes = ckpt.get("n_action_classes", len(verb_names))
n_noun_classes = ckpt.get("n_noun_classes", len(noun_names) if noun_names is not None else 50)
n_hand_classes = ckpt.get("n_hand_classes", len(hand_names))

print("Nombre de verbes :", n_verb_classes)
print("Nombre d'objets/noms :", n_noun_classes)
print("Nombre de classes main :", n_hand_classes)
print("Verbes :", verb_names)

Clés du checkpoint : dict_keys(['epoch', 'model', 'optimizer', 'scheduler', 'best_vmca', 'n_action_classes', 'n_noun_classes', 'verb_names', 'noun_names', 'model_name'])
Nombre de verbes : 30
Nombre d'objets/noms : 50
Nombre de classes main : 3
Verbes : ['take', 'put', 'wash', 'open', 'close', 'turn-on', 'cut', 'turn-off', 'pour', 'mix', 'move', 'remove', 'throw', 'dry', 'shake', 'scoop', 'adjust', 'squeeze', 'press', 'flip', 'turn', 'check', 'scrub', 'pull', 'pat', 'lift', 'hold', 'drop', 'transition', 'reach']


## 6. Reconstruire le modèle multi-têtes VideoMAE

Le dépôt GitHub indiqué par la fiche HF n'étant pas accessible publiquement dans notre cas, on reconstruit ici un wrapper simple :

- backbone VideoMAE ;
- tête verbe ;
- tête main ;
- tête objet/noun.

La cellule charge ensuite les poids du checkpoint avec `strict=False` pour rester robuste aux noms de clés.

In [7]:
import torch.nn as nn

class VideoMAEMultiHead(nn.Module):
    def __init__(self, base_model_id, n_verb_classes, n_hand_classes, n_noun_classes, dropout=0.1):
        super().__init__()
        self.backbone = VideoMAEModel.from_pretrained(base_model_id)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.verb_head = nn.Linear(hidden_size, n_verb_classes)
        self.hand_head = nn.Linear(hidden_size, n_hand_classes)
        self.noun_head = nn.Linear(hidden_size, n_noun_classes)

    def forward(self, pixel_values):
        # pixel_values attendu : (B, T, C, H, W)
        outputs = self.backbone(pixel_values=pixel_values)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        verb_logits = self.verb_head(pooled)
        hand_logits = self.hand_head(pooled)
        noun_logits = self.noun_head(pooled)
        return verb_logits, hand_logits, noun_logits

model = VideoMAEMultiHead(
    BASE_MODEL_ID,
    n_verb_classes=n_verb_classes,
    n_hand_classes=n_hand_classes,
    n_noun_classes=n_noun_classes,
    dropout=0.1
)

state = ckpt.get("model", ckpt)

# Plusieurs checkpoints utilisent des préfixes différents. On tente quelques variantes.
def try_load_state_dict(model, state):
    attempts = []
    attempts.append(("original", state))
    attempts.append(("strip_module", {k.replace("module.", ""): v for k, v in state.items()}))
    attempts.append(("rename_model_to_backbone", {k.replace("model.", "backbone."): v for k, v in state.items()}))
    attempts.append(("rename_videomae_to_backbone", {k.replace("videomae.", "backbone."): v for k, v in state.items()}))

    best = None
    for name, sd in attempts:
        try:
            result = model.load_state_dict(sd, strict=False)
            missing = len(result.missing_keys)
            unexpected = len(result.unexpected_keys)
            score = missing + unexpected
            print(f"Tentative {name}: missing={missing}, unexpected={unexpected}")
            if best is None or score < best[0]:
                best = (score, name, result)
        except Exception as e:
            print(f"Tentative {name} échouée:", e)

    print("Meilleure tentative :", best[1])
    print("Missing keys exemple :", best[2].missing_keys[:10])
    print("Unexpected keys exemple :", best[2].unexpected_keys[:10])
    return best

best_load = try_load_state_dict(model, state)
model = model.to(DEVICE)
model.eval()

print("Modèle prêt sur", DEVICE)

config.json:   0%|          | 0.00/21.3k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/346M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/345M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

[transformers] VideoMAEModel LOAD REPORT from: MCG-NJU/videomae-base-finetuned-ssv2
Key                                                        | Status     | 
-----------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias | UNEXPECTED | 
classifier.weight                                          | UNEXPECTED | 
fc_norm.weight                                             | UNEXPECTED | 
classifier.bias                                            | UNEXPECTED | 
fc_norm.bias                                               | UNEXPECTED | 
encoder.layer.{0...11}.attention.attention.query.bias      | MISSING    | 
encoder.layer.{0...11}.attention.attention.key.bias        | MISSING    | 
encoder.layer.{0...11}.attention.attention.value.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Tentative original: missing=194, unexpected=184
Tentative strip_module: missing=194, unexpected=184
Tentative rename_model_to_backbone: missing=194, unexpected=184
Tentative rename_videomae_to_backbone: missing=194, unexpected=184
Meilleure tentative : original
Missing keys exemple : ['backbone.embeddings.patch_embeddings.projection.weight', 'backbone.embeddings.patch_embeddings.projection.bias', 'backbone.encoder.layer.0.attention.attention.query.weight', 'backbone.encoder.layer.0.attention.attention.query.bias', 'backbone.encoder.layer.0.attention.attention.key.weight', 'backbone.encoder.layer.0.attention.attention.key.bias', 'backbone.encoder.layer.0.attention.attention.value.weight', 'backbone.encoder.layer.0.attention.attention.value.bias', 'backbone.encoder.layer.0.attention.output.dense.weight', 'backbone.encoder.layer.0.attention.output.dense.bias']
Unexpected keys exemple : ['encoder.embeddings.patch_embeddings.projection.weight', 'encoder.embeddings.patch_embeddings.projectio

## 7. Fonctions vidéo : lecture, preprocessing, prédiction

In [8]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def sample_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Impossible d'ouvrir la vidéo : {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        raise ValueError(f"Nombre de frames invalide pour : {video_path}")

    indices = np.linspace(0, total_frames - 1, num_frames).astype(int)
    frames = []

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame).resize((224, 224))
        frames.append(frame)

    cap.release()

    if len(frames) == 0:
        raise ValueError("Aucune frame lue. Vérifie le chemin ou le codec vidéo.")

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return frames[:num_frames]


def frames_to_tensor(frames):
    tensors = []
    for img in frames:
        arr = np.array(img).astype(np.float32) / 255.0
        x = torch.from_numpy(arr).permute(2, 0, 1)
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        tensors.append(x)
    # (1, T, C, H, W)
    return torch.stack(tensors, dim=0).unsqueeze(0)


def topk_from_logits(logits, names, k=3):
    probs = torch.softmax(logits, dim=-1)[0]
    k = min(k, probs.shape[0])
    values, indices = torch.topk(probs, k=k)
    out = []
    for score, idx in zip(values, indices):
        idx = int(idx.item())
        label = names[idx] if names is not None and idx < len(names) else str(idx)
        out.append({"label": label, "score": float(score.item())})
    return out


def predict_clip(video_path, topk=3):
    frames = sample_frames(video_path, num_frames=16)
    pixel_values = frames_to_tensor(frames).to(DEVICE)

    with torch.no_grad():
        verb_logits, hand_logits, noun_logits = model(pixel_values)

    verbs = topk_from_logits(verb_logits, verb_names, k=topk)
    hands = topk_from_logits(hand_logits, hand_names, k=topk)
    nouns = topk_from_logits(noun_logits, noun_names, k=topk) if noun_names is not None else topk_from_logits(noun_logits, None, k=topk)

    return {
        "verb_topk": verbs,
        "hand_topk": hands,
        "noun_topk": nouns,
        "best_verb": verbs[0]["label"],
        "best_verb_score": verbs[0]["score"],
        "best_hand": hands[0]["label"],
        "best_hand_score": hands[0]["score"],
        "best_noun": nouns[0]["label"],
        "best_noun_score": nouns[0]["score"],
        "best_action": f"{verbs[0]['label']} {nouns[0]['label']}"
    }

## 8. Envoyer une vidéo dans Colab

In [9]:
uploaded = files.upload()

videos = glob.glob("*.mp4") + glob.glob("*.MP4") + glob.glob("*.mov") + glob.glob("*.MOV") + glob.glob("*.avi")
print("Vidéos trouvées :", videos)

if len(videos) == 0:
    raise ValueError("Aucune vidéo trouvée. Upload une vidéo .mp4, .mov ou .avi.")

video_path = videos[0]
print("Vidéo utilisée :", video_path)

Saving P02_101.mp4 to P02_101.mp4
Vidéos trouvées : ['P02_101.mp4']
Vidéo utilisée : P02_101.mp4


## 9. Tester une prédiction sur toute la vidéo comme un seul clip

In [10]:
test_pred = predict_clip(video_path, topk=3)
test_pred

{'verb_topk': [{'label': 'adjust', 'score': 0.9083491563796997},
  {'label': 'squeeze', 'score': 0.04047837853431702},
  {'label': 'hold', 'score': 0.03841230645775795}],
 'hand_topk': [{'label': 'right', 'score': 0.9985734224319458},
  {'label': 'both', 'score': 0.0014264833880588412},
  {'label': 'left', 'score': 4.506318518338048e-08}],
 'noun_topk': [{'label': 'water', 'score': 0.740605354309082},
  {'label': 'glass', 'score': 0.18607836961746216},
  {'label': 'mushroom', 'score': 0.05093502625823021}],
 'best_verb': 'adjust',
 'best_verb_score': 0.9083491563796997,
 'best_hand': 'right',
 'best_hand_score': 0.9985734224319458,
 'best_noun': 'water',
 'best_noun_score': 0.740605354309082,
 'best_action': 'adjust water'}

## 10. Créer une vidéo annotée segment par segment

Par défaut, le modèle prédit toutes les **5 secondes**. Tu peux changer `WINDOW_SECONDS = 5` vers `2`, `3`, etc.

In [11]:
def make_temp_clip(input_video, output_clip, start_frame, end_frame, fps, width, height):
    cap = cv2.VideoCapture(input_video)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_clip, fourcc, fps, (width, height))

    current = int(start_frame)
    while current < int(end_frame):
        ret, frame = cap.read()
        if not ret:
            break
        out.write(frame)
        current += 1

    cap.release()
    out.release()


def draw_text_box(frame, lines, x=20, y=20):
    line_h = 32
    box_w = min(frame.shape[1] - 40, 980)
    box_h = 25 + line_h * len(lines)
    cv2.rectangle(frame, (x, y), (x + box_w, y + box_h), (0, 0, 0), -1)
    for i, line in enumerate(lines):
        yy = y + 38 + i * line_h
        cv2.putText(frame, line, (x + 15, yy), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
    return frame


def create_annotated_video(input_video, output_video="hf_videomae_annotated.mp4", window_seconds=5, topk=3):
    cap = cv2.VideoCapture(input_video)
    if not cap.isOpened():
        raise ValueError(f"Impossible d'ouvrir : {input_video}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 25

    frames_per_window = max(1, int(window_seconds * fps))
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_video, fourcc, fps, (width, height))

    rows = []
    current_pred = None
    frame_idx = 0

    pbar = tqdm(total=total_frames, desc="Annotation vidéo")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frames_per_window == 0:
            start_frame = frame_idx
            end_frame = min(frame_idx + frames_per_window, total_frames)
            temp_clip = "__temp_clip_hf.mp4"

            make_temp_clip(input_video, temp_clip, start_frame, end_frame, fps, width, height)

            try:
                current_pred = predict_clip(temp_clip, topk=topk)
                row = {
                    "start_second": round(start_frame / fps, 2),
                    "end_second": round(end_frame / fps, 2),
                    "best_action": current_pred["best_action"],
                    "best_verb": current_pred["best_verb"],
                    "verb_score": current_pred["best_verb_score"],
                    "best_hand": current_pred["best_hand"],
                    "hand_score": current_pred["best_hand_score"],
                    "best_noun": current_pred["best_noun"],
                    "noun_score": current_pred["best_noun_score"],
                    "verb_topk": json.dumps(current_pred["verb_topk"], ensure_ascii=False),
                    "hand_topk": json.dumps(current_pred["hand_topk"], ensure_ascii=False),
                    "noun_topk": json.dumps(current_pred["noun_topk"], ensure_ascii=False),
                }
            except Exception as e:
                current_pred = None
                row = {
                    "start_second": round(start_frame / fps, 2),
                    "end_second": round(end_frame / fps, 2),
                    "error": str(e)
                }

            rows.append(row)

            if os.path.exists(temp_clip):
                os.remove(temp_clip)

        if current_pred is not None:
            lines = [
                f"Action: {current_pred['best_action']}",
                f"Verb: {current_pred['best_verb']}  score={current_pred['best_verb_score']:.2f}",
                f"Noun: {current_pred['best_noun']}  score={current_pred['best_noun_score']:.2f}",
                f"Hand: {current_pred['best_hand']}  score={current_pred['best_hand_score']:.2f}",
            ]
        else:
            lines = ["Prediction: processing / error"]

        frame = draw_text_box(frame, lines)
        out.write(frame)
        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    out.release()

    df = pd.DataFrame(rows)
    csv_path = "hf_videomae_predictions.csv"
    df.to_csv(csv_path, index=False)

    return output_video, csv_path, df

WINDOW_SECONDS = 5
annotated_path, csv_path, df_pred = create_annotated_video(
    input_video=video_path,
    output_video="hf_videomae_annotated.mp4",
    window_seconds=WINDOW_SECONDS,
    topk=3
)

df_pred.head()

Annotation vidéo:   0%|          | 0/12186 [00:00<?, ?it/s]

,start_second,end_second,best_action,best_verb,verb_score,best_hand,hand_score,best_noun,noun_score,verb_topk,hand_topk,noun_topk
0,0.0,5.0,turn-off phone,turn-off,0.984120,right,0.883696,phone,0.960890,"[{""label"": ""turn-off"", ""score"": 0.984120190143...","[{""label"": ""right"", ""score"": 0.883695602416992...","[{""label"": ""phone"", ""score"": 0.960889518260955..."
1,5.0,10.0,scoop knife,scoop,0.809936,right,0.850167,knife,0.915780,"[{""label"": ""scoop"", ""score"": 0.809935748577117...","[{""label"": ""right"", ""score"": 0.850167095661163...","[{""label"": ""knife"", ""score"": 0.915779709815979..."
2,10.0,15.0,scoop garlic,scoop,0.437639,right,0.965083,garlic,0.871533,"[{""label"": ""scoop"", ""score"": 0.437638789415359...","[{""label"": ""right"", ""score"": 0.965082585811615...","[{""label"": ""garlic"", ""score"": 0.87153315544128..."
3,15.0,20.0,remove garlic,remove,0.998907,right,0.982650,garlic,0.994416,"[{""label"": ""remove"", ""score"": 0.99890708923339...","[{""label"": ""right"", ""score"": 0.982650339603424...","[{""label"": ""garlic"", ""score"": 0.99441564083099..."
4,20.0,25.0,put glass,put,0.880234,both,0.990567,glass,0.454069,"[{""label"": ""put"", ""score"": 0.8802341818809509}...","[{""label"": ""both"", ""score"": 0.990567147731781}...","[{""label"": ""glass"", ""score"": 0.454069495201110..."


## 11. Afficher la vidéo annotée dans Colab

In [ ]:
mp4 = open(annotated_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f'''
<video width="900" controls>
    <source src="{data_url}" type="video/mp4">
</video>
''')

## 12. Télécharger la vidéo annotée et le CSV

In [12]:
files.download(annotated_path)
files.download(csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Notes

- Le modèle travaille sur des clips de 16 frames en 224×224.
- Pour une vidéo longue, on prédit une action par fenêtre temporelle, par exemple 5 secondes.
- Si les résultats sont instables, augmente `WINDOW_SECONDS` à 7 ou 10 secondes.
- Si tu veux une annotation plus fréquente, diminue à 2 ou 3 secondes, mais les scores peuvent devenir moins fiables.